## 청소기 테이블 재구성

In [1]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 경로 설정
# =========================
input_path = Path(r"C:/project_skn/project4/4th_project/products/data/preprocessing/vacuum/lge_vacuum_all_products.csv")
output_path = Path("ProductVAC.csv")

# =========================
# 변환 함수
# =========================
def blank_if_na(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def clean_float(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+(?:\.\d+)?", text)

    if not nums:
        return pd.NA

    return float(nums[0])


def clean_proficiency(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "해당없음" in text or "해당 없음" in text or "비대상" in text or "대상 외" in text:
        return pd.NA

    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def split_size(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA, pd.NA

    text = str(value).replace(",", "").replace('"', "").strip()
    parts = re.split(r"\s*[xX×]\s*", text)

    if len(parts) < 3:
        return pd.NA, pd.NA, pd.NA

    width = clean_int(parts[0])
    height = clean_int(parts[1])
    depth = clean_int(parts[2])

    return width, height, depth


def clean_battery_cnt(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).split("(")[0].strip()
    return clean_int(text)


# =========================
# 원본 읽기
# =========================
df = pd.read_csv(input_path)

# =========================
# 크기 분리
# =========================
unit_size = df["크기(청소기 본체)"].apply(split_size)
tower_size = df["크기(올인원 타워)"].apply(split_size)

df["unit_width(mm)"] = unit_size.apply(lambda x: x[0])
df["unit_height(mm)"] = unit_size.apply(lambda x: x[1])
df["unit_depth(mm)"] = unit_size.apply(lambda x: x[2])

df["tower_width(mm)"] = tower_size.apply(lambda x: x[0])
df["tower_height(mm)"] = tower_size.apply(lambda x: x[1])
df["tower_depth(mm)"] = tower_size.apply(lambda x: x[2])

# =========================
# 새 DataFrame 구성
# =========================
new_df = pd.DataFrame({
    "product_code": [f"VAC{i:03d}" for i in range(len(df))],
    "name": df["제품명"].apply(blank_if_na),
    "img_link": df["이미지 링크"].apply(blank_if_na),
    "price": df["가격(원)"].apply(clean_int),
    "proficiency_level": df["에너지 소비효율 등급"].apply(clean_proficiency),
    "power_consum(W)": df["정격소비전력(W)"].apply(clean_int),
    "unit_width(mm)": df["unit_width(mm)"],
    "unit_height(mm)": df["unit_height(mm)"],
    "unit_depth(mm)": df["unit_depth(mm)"],
    "tower_width(mm)": df["tower_width(mm)"],
    "tower_height(mm)": df["tower_height(mm)"],
    "tower_depth(mm)": df["tower_depth(mm)"],
    "unit_weight(kg)": df["무게(청소기 본체)"].apply(clean_float),
    "tower_weight(kg)": df["무게(올인원 타워)"].apply(clean_float),
    "color": df["색상(청소기 본체)"].apply(blank_if_na),
    "suction_power(W)": df["최대 흡입력(W)"].apply(clean_int),
    "battery_cnt(개)": df["배터리 포함 수(개)"].apply(clean_battery_cnt),
    "manual_link": df["제품 사용 설명서"].apply(blank_if_na)
})

# =========================
# 타입 지정
# =========================
str_cols = [
    "product_code", "name", "img_link", "color", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "power_consum(W)",
    "unit_width(mm)", "unit_height(mm)", "unit_depth(mm)",
    "tower_width(mm)", "tower_height(mm)", "tower_depth(mm)",
    "suction_power(W)", "battery_cnt(개)"
]

float_cols = [
    "unit_weight(kg)", "tower_weight(kg)"
]

for col in str_cols:
    new_df[col] = new_df[col].astype("string")

for col in int_cols:
    new_df[col] = new_df[col].astype("Int64")

for col in float_cols:
    new_df[col] = new_df[col].astype("Float64")

# =========================
# 저장
# =========================
new_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print(new_df.head())
print(f"저장 완료: {output_path}")

  product_code                    name  \
0       VAC000  LG 코드제로 오브제컬렉션 A7 Core   
1       VAC001  LG 코드제로 오브제컬렉션 A7 Core   
2       VAC002  LG 코드제로 오브제컬렉션 A7 Core   
3       VAC003  LG 코드제로 오브제컬렉션 A7 Core   
4       VAC004    LG 코드제로 AI 오브제컬렉션 A9   

                                            img_link    price  \
0  https://www.lge.co.kr/kr/images/vacuum-cleaner...   900000   
1  https://www.lge.co.kr/kr/images/vacuum-cleaner...     <NA>   
2  https://www.lge.co.kr/kr/images/vacuum-cleaner...  1041000   
3  https://www.lge.co.kr/kr/images/vacuum-cleaner...  1130000   
4  https://www.lge.co.kr/kr/images/vacuum-cleaner...  1331000   

   proficiency_level  power_consum(W)  unit_width(mm)  unit_height(mm)  \
0               <NA>              350             250             1120   
1               <NA>              350             250             1120   
2               <NA>              350             250             1120   
3               <NA>              350             250       